In [ ]:
import subprocess
import sys
ULTRALYTICS_VERSION = "8.4.120"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics==" + ULTRALYTICS_VERSION, "gradio"], check=True)
try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False
print("Running on Colab:", RUNNING_ON_COLAB)

Running on Colab: True


In [ ]:
import json
from pathlib import Path
import io
import os
import time

import gradio as gr
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

print("gradio:", gr.__version__)

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    PROJECT_ROOT = str(Path.cwd() / "C389")

WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "L3", "best.pt")
print("weights:", WEIGHTS)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
gradio: 6.31.0
Mounted at /content/drive
weights: /content/drive/MyDrive/C389/artifacts/L3/best.pt


In [ ]:
if not os.path.isfile(WEIGHTS):
    raise FileNotFoundError(
        f"Trained weights not found: {WEIGHTS}\n"
        "In Lesson 3, finish training and run the final save cell (Section 9). "
        "Then return here using the same Google Drive account and rerun this cell."
    )
MODEL_PATH = WEIGHTS
USING_TRAINED_MODEL = True
METRICS_PATH = os.path.join(PROJECT_ROOT, "artifacts", "L3", "baseline_metrics.json")
if not os.path.isfile(METRICS_PATH):
    raise FileNotFoundError(f"Missing {METRICS_PATH}. Run Lesson 3's final save cell.")
with open(METRICS_PATH, encoding="utf-8") as handle:
    TRAINING_METADATA = json.load(handle)
print("Found Lesson 3 model and metadata. No retraining needed.")

Found Lesson 3 model and metadata. No retraining needed.


In [ ]:
MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]
EXPECTED_CLASSES = ["person_left", "person_left_wheel", "person_right", "person_right_wheel"]
if CLASS_NAMES != EXPECTED_CLASSES:
    raise ValueError(f"Wrong model classes: {CLASS_NAMES}. Expected {EXPECTED_CLASSES}.")
if TRAINING_METADATA.get("classes") != CLASS_NAMES:
    raise ValueError("The saved metrics and model have different classes. Save both again in Lesson 3.")
CONFIDENCE_THRESHOLD = float(TRAINING_METADATA.get("confidence_threshold", 0.40))
MAX_IMAGE_PIXELS = 40_000_000
SERVICE_VERSION = "1.0.0-ca1"
print("model loaded from:", MODEL_PATH)
print("classes:", dict(enumerate(CLASS_NAMES)))
print("confidence threshold:", CONFIDENCE_THRESHOLD)

model loaded from: /content/drive/MyDrive/C389/artifacts/L3/best.pt
classes: {0: 'person_left', 1: 'person_left_wheel', 2: 'person_right', 3: 'person_right_wheel'}
confidence threshold: 0.4


In [ ]:
def validate_image(uploaded):
    """Check that what arrived can be turned into an image to run on.

    Args:
        uploaded: whatever the caller sent. A path, a file like object, a PIL
            image, or None when nothing was sent at all

    Returns:
        (image, error), exactly one of which is None. The error is a dict
        carrying a status code and a message the caller could act on

    Requirement:
        Four cases are refused before any inference runs: nothing sent, an empty
        file, a file that is not a readable image, and an image so large it would
        exhaust memory.

        A service that hands a text file to the model and returns a stack trace
        has told the caller nothing. Every refusal here names what was wrong and
        what to send instead.
    """
    if uploaded is None:
        return None, {"status": 400, "error": "No image was sent.",
                      "detail": "Attach a JPEG or PNG and send it again."}

    try:
        if isinstance(uploaded, Image.Image):
            image = uploaded
        elif isinstance(uploaded, (bytes, bytearray)):
            if not uploaded:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file carried no data."}
            image = Image.open(io.BytesIO(uploaded))
            image.load()
        else:
            if os.path.getsize(uploaded) == 0:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file '%s' is zero bytes."
                                        % os.path.basename(str(uploaded))}
            image = Image.open(uploaded)
            image.load()
    except (UnidentifiedImageError, OSError):
        return None, {"status": 415, "error": "That file is not a readable image.",
                      "detail": "Send a JPEG or PNG."}

    if image.width * image.height > MAX_IMAGE_PIXELS:
        return None, {"status": 413, "error": "The image is too large to process.",
                      "detail": "%d by %d pixels is beyond the limit of %d pixels."
                                % (image.width, image.height, MAX_IMAGE_PIXELS)}

    return image.convert("RGB"), None

In [ ]:
def count_people(image, threshold):
    """Run the model on one image and total the detections by class.

    Args:
        image: a PIL image, already validated
        threshold: detections below this confidence are dropped

    Returns:
        (counts, kept), where counts maps nested categories to whole numbers

    Requirement:
        The same rule as Lesson 3. Every class appears in counts, including the
        ones at zero, and an image with nobody in it is a valid result of zero
        rather than an error.
    """
    result = MODEL.predict(image, conf=0.01, verbose=False)[0]

    kept = []
    boxes = result.boxes
    if boxes is not None:
        for index in range(len(boxes)):
            confidence = round(float(boxes.conf[index].item()), 3)
            if confidence < threshold:
                continue
            class_id = int(boxes.cls[index].item())
            kept.append({
                "class_name": CLASS_NAMES[class_id],
                "confidence": confidence,
                "box": [round(float(v), 1) for v in boxes.xyxy[index].tolist()],
            })

    counts = {
        "pedestrian": {
            "total": 0,
            "person_left": 0,
            "person_right": 0
        },
        "wheeled": {
            "total": 0,
            "person_left_wheel": 0,
            "person_right_wheel": 0
        }
    }

    for detection in kept:
        name = detection["class_name"]
        if "wheel" in name:
            counts["wheeled"][name] = counts["wheeled"].get(name, 0) + 1
            counts["wheeled"]["total"] += 1
        else:
            counts["pedestrian"][name] = counts["pedestrian"].get(name, 0) + 1
            counts["pedestrian"]["total"] += 1

    return counts, kept

In [ ]:
def build_reply(uploaded, threshold=CONFIDENCE_THRESHOLD):
    """The whole service, in one function. Validate, count, answer.

    Returns a dict either way. A caller never has to tell success from failure by
    reading prose: a reply carrying "error" failed, and one carrying "counts" did
    not.

    A CA1 twist that adds a class or an extra field changes this function and
    nothing else, which is why the reply is built in one place.
    """
    image, error = validate_image(uploaded)
    if error is not None:
        return error, None

    started = time.time()
    counts, detections = count_people(image, threshold)
    elapsed = time.time() - started

    reply = {
        "status": 200,
        "counts": counts,
        "confidence_threshold": threshold,
        "model_version": SERVICE_VERSION,
        "trained_model": USING_TRAINED_MODEL,
        "image_size": [image.width, image.height],
        "inference_seconds": round(elapsed, 3),
        "detections": detections,
    }
    # Printed rather than kept, so the Colab output below the cell is this
    # service's log. Section 6 reads it.
    print("counted %s in %.3fs on a %dx%d image"
          % (counts, elapsed, image.width, image.height))
    return reply, image

In [ ]:
def annotate(image, detections):
    """Draw the kept boxes, so the reply can be checked by eye as well as read."""
    from PIL import ImageDraw

    marked = image.copy()
    draw = ImageDraw.Draw(marked)
    for detection in detections:
        left, top, right, bottom = detection["box"]
        colour = (214, 69, 65) if detection["class_name"].startswith("person_left") else (46, 112, 196)
        draw.rectangle([left, top, right, bottom], outline=colour, width=3)
        draw.text((left + 3, max(0, top - 12)),
                  "%s %.2f" % (detection["class_name"], detection["confidence"]),
                  fill=colour)
    return marked


def predict(uploaded, threshold):
    """What the page calls. Returns the annotated image and the reply."""
    reply, image = build_reply(uploaded, threshold)
    if image is None:
        return None, reply
    return annotate(image, reply["detections"]), reply

In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Walkway photograph"),
        gr.Slider(0.05, 0.95, value=CONFIDENCE_THRESHOLD, step=0.05,
                  label="Confidence threshold"),
    ],
    outputs=[
        gr.Image(label="Detections"),
        gr.JSON(label="Reply"),
    ],
    title="People Counting Assistant",
    description="Upload one photograph. The reply says how many people are "
                "in each of the four trained classes: person_left, person_left_wheel, "
                "person_right and person_right_wheel. Counts are per image.",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://869c6f913f157f862c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import tempfile

scratch = tempfile.mkdtemp()

text_file = os.path.join(scratch, "not_an_image.txt")
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write("this is not a photograph")

empty_file = os.path.join(scratch, "empty.jpg")
open(empty_file, "wb").close()

huge = Image.new("RGB", (9000, 7000), (200, 200, 200))
huge_file = os.path.join(scratch, "huge.jpg")
huge.save(huge_file, quality=30)

nobody = Image.new("RGB", (640, 480), (214, 210, 202))
nobody_file = os.path.join(scratch, "nobody.jpg")
nobody.save(nobody_file)

CASES = [
    ("nothing sent", None),
    ("a text file", text_file),
    ("an empty file", empty_file),
    ("a very large image", huge_file),
    ("an image with nobody in it", nobody_file),
]

# Collected first, printed afterwards. The service logs a line for every call it
# serves, and a library may print a one-time message the first time it is asked
# for something, so printing the table as it goes puts that chatter through the
# middle of the thing being read.
rows = []
for label, payload in CASES:
    reply, _ = build_reply(payload)
    summary = reply.get("error") or ("counts " + str(reply.get("counts")))
    rows.append((label, reply["status"], summary))

print("")
print("%-28s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-28s %-7s %s" % (label, status, summary))

requirements: Ultralytics requirement ['pi-heif'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 228ms
Prepared 1 package in 44ms
Installed 1 package in 1ms
 + pi-heif==1.4.0

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

counted {'person_left': 0, 'person_left_wheel': 0, 'person_right': 0, 'person_right_wheel': 0} in 2.670s on a 640x480 image

case                         status  what came back
nothing sent                 400     No image was sent.
a text file                  415     That file is not a readable image.
an empty file                400     The upload was empty.
a very large image           413     The image is too large to process.
an image with nobody in it   200     counts {'person_left': 0, 'person_left_wheel': 0, 'person_right': 0, 'person_right_wheel': 0}


In [ ]:
# Warm, measured here, where this process is exactly the warm service.
sample = Image.new("RGB", (640, 480), (214, 210, 202))

warm = []
for attempt in range(5):
    started = time.time()
    count_people(sample, CONFIDENCE_THRESHOLD)
    warm.append(time.time() - started)

warm_average = sum(warm) / len(warm)
print("warm inference, average : %.3f s" % warm_average)

warm inference, average : 0.017 s


In [ ]:
import subprocess
import sys

# A separate process, so nothing this notebook has already done counts. It
# imports, loads the weights and answers one request, and reports where its own
# time went. The figure that matters is the total measured from out here, because
# that is what the caller waits through.
weights = os.path.abspath(MODEL_PATH) if os.path.exists(MODEL_PATH) else MODEL_PATH
import tempfile
probe_path = os.path.join(tempfile.mkdtemp(prefix="ca1_probe_"), "cold_start_probe.py")

with open(probe_path, "w", encoding="utf-8") as handle:
    print("import sys, time", file=handle)
    print("launch = float(sys.argv[1])", file=handle)
    print("start = time.time()", file=handle)
    print("from ultralytics import YOLO", file=handle)
    print("from PIL import Image", file=handle)
    print("imported = time.time()", file=handle)
    print("model = YOLO(%r)" % weights, file=handle)
    print("loaded = time.time()", file=handle)
    print("model.predict(Image.new('RGB', (640, 480)), conf=0.01, verbose=False)",
          file=handle)
    print("answered = time.time()", file=handle)
    print("print('  importing the libraries : %.3f s' % (imported - start))", file=handle)
    print("print('  reading the weights     : %.3f s' % (loaded - imported))", file=handle)
    print("print('  answering the first time: %.3f s' % (answered - loaded))", file=handle)
    print("print('ANSWERED %.3f' % (answered - launch))", file=handle)

started = time.time()
probe = subprocess.run([sys.executable, probe_path, repr(started)],
                       capture_output=True, text=True)
process_lifetime = time.time() - started

# The caller waits from the moment the process is launched until it answers. What
# happens after that, releasing the GPU and shutting down, can take seconds and
# nobody is waiting for it, so timing the whole subprocess overstates the wait.
cold_total = process_lifetime
report = []
for line in probe.stdout.splitlines():
    if not line.strip():
        continue
    if line.startswith("ANSWERED "):
        cold_total = float(line.split()[1])
    else:
        report.append(line)

if probe.returncode != 0:
    print("The probe did not run. Its last words:")
    print(probe.stderr.strip()[-600:])
else:
    print("inside a process that has just started:")
    for line in report:
        print(line)
    print("")
    print("somebody arriving at a cold service waits : %.3f s" % cold_total)
    print("somebody arriving at a warm one waits     : %.3f s" % warm_average)
    print("cold is %.0f times the warm figure" % (cold_total / warm_average))
    print("")
    print("the probe process lived for %.3f s in total. The difference is it"
          % process_lifetime)
    print("shutting down and releasing the GPU, which no caller waits for.")

inside a process that has just started:
  importing the libraries : 2.298 s
  reading the weights     : 0.078 s
  answering the first time: 1.005 s

somebody arriving at a cold service waits : 3.428 s
somebody arriving at a warm one waits     : 0.017 s
cold is 197 times the warm figure

the probe process lived for 6.058 s in total. The difference is it
shutting down and releasing the GPU, which no caller waits for.


In [ ]:
# Leave False while using the demo. Set True and run this cell when finished.
CLOSE_DEMO = False
if CLOSE_DEMO:
    demo.close()
    print("The share link is closed.")
else:
    print("Demo left running. To stop it, set CLOSE_DEMO = True and rerun this cell.")

Demo left running. To stop it, set CLOSE_DEMO = True and rerun this cell.
